In [2]:
!pip install langchain langchain-community transformers torch sentence-transformers faiss-cpu pypdf

   ---------------------------------------- 0.0/570.8 kB ? eta -:--:--
   ------------------------------------ --- 524.3/570.8 kB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 570.8/570.8 kB 1.5 MB/s  0:00:00



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
!pip install -U langchain langchain-community langchain-classic transformers torch sentence-transformers faiss-cpu pypdf

  Using cached torch-2.11.0-cp312-cp312-win_amd64.whl.metadata (29 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/10.2 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.2 MB 3.4 MB/s eta 0:00:03
   -------- ------------------------------- 2.1/10.2 MB 5.3 MB/s eta 0:00:02
   ------------ --------------------------- 3.1/10.2 MB 5.4 MB/s eta 0:00:02
   ----------------- ---------------------- 4.5/10.2 MB 5.7 MB/s eta 0:00:02
   ---------------------- ----------------- 5.8/10.2 MB 6.1 MB/s eta 0:00:01
   ---------------------------- ----------- 7.3/10.2 MB 6.2 MB/s eta 0:00:01
   ---------------------------------- ----- 8.9/10.2 MB 6.4 MB/s eta 0:00:01
   ---------------------------------------  10.2/10.2 MB 6.4 MB/s eta 0:00:01
   ---------------------------------------- 10.2/10.2 MB 6.3 MB/s  0:00:01
Using cached torch-2.11.0-cp312-cp312-win_amd64.whl (114.6 MB)
Using cached sympy-1.14.0-py3-none-any.whl

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.5.1+cu121 requires torch==2.5.1+cu121, but you have torch 2.11.0 which is incompatible.
torchvision 0.20.1+cu121 requires torch==2.5.1+cu121, but you have torch 2.11.0 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline
from langchain_core.prompts import PromptTemplate
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.tools import tool
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
import os

In [10]:
text_generator = pipeline(
    task="text-generation",
    model="distilgpt2",
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7
)

llm = HuggingFacePipeline(pipeline=text_generator)
print("Model loaded successfully!\n")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Model loaded successfully!



In [11]:
# 1. BASIC LLM CALL

print("===== 1. Basic LLM Call =====")
response = llm.invoke("Explain LangChain in simple words.")
print(response)
print("\n")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== 1. Basic LLM Call =====
Explain LangChain in simple words.








































































































In [12]:
# 2. PROMPT TEMPLATE

prompt = PromptTemplate.from_template("Explain {topic} to a beginner in simple terms.")

formatted_prompt = prompt.invoke({"topic": "vector databases"})
print(formatted_prompt.text)
print("\n")

Explain vector databases to a beginner in simple terms.




In [13]:
# 3. SIMPLE CHAIN

chain_prompt = PromptTemplate.from_template("Give a concise explanation of {topic}.")

chain = chain_prompt | llm
chain_response = chain.invoke({"topic": "LangChain agents"})
print(chain_response)
print("\n")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Give a concise explanation of LangChain agents.




If you like this article or would like to give a shout to the author, he may visit the link below.




In [14]:
# 4. MEMORY EXAMPLE

memory = ConversationBufferMemory()

memory.save_context(
    {"input": "My name is Rahul"},
    {"output": "Nice to meet you Rahul"}
)

memory.save_context(
    {"input": "I like Formula 1"},
    {"output": "That's exciting!"}
)

history = memory.load_memory_variables({})
print(history)
print("\n")

{'history': "Human: My name is Rahul\nAI: Nice to meet you Rahul\nHuman: I like Formula 1\nAI: That's exciting!"}




C:\Users\Vidhi\AppData\Local\Temp\ipykernel_34996\3830545652.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()


In [15]:
# 5. TOOL EXAMPLE

@tool
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

result = multiply_numbers.invoke({"a": 5, "b": 6})
print("Tool Output:", result)
print("\n")

Tool Output: 30




In [16]:
# 6. PDF LOADER EXAMPLE

pdf_path = "sample.pdf"

if os.path.exists(pdf_path):
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    print("Loaded PDF successfully")
    print(docs[0].page_content[:500])
else:
    print("sample.pdf not found. Place a sample PDF in the folder to test this section.")

print("\n")

sample.pdf not found. Place a sample PDF in the folder to test this section.




In [17]:
# 7. VECTOR STORE / FAISS EXAMPLE

texts = [
    "LangChain helps build modular LLM applications.",
    "Vector stores enable semantic similarity search.",
    "Memory helps retain previous conversation context."
]

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_db = FAISS.from_texts(texts, embeddings)

results = vector_db.similarity_search("How does semantic search work?")

print("Most relevant result:")
print(results[0].page_content)
print("\n")

C:\Users\Vidhi\AppData\Local\Temp\ipykernel_34996\2771252739.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Vidhi\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vidhi\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Most relevant result:
Vector stores enable semantic similarity search.


